Primero hacemos una prueba de que la importación de los datos se hace de forma correcta desde el disco HHD `D:` usando las direcciónes del `.env` y de `config.py`, y que las funciones del modulo `loader.py` funcionan correctamente

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import sys
import os

# 1. Cargar .env desde la raíz del proyecto
project_root = Path.cwd().parents[0]
load_dotenv(project_root / ".env")

# 2. Añadir raíz del proyecto al sys.path
sys.path.insert(0, str(project_root))

# 3. Importar loader que importa de config.py correctamente las rutas
from src.data.loader import get_quarters, load_quarter, load_failures

# 4. Verificar
from src.utils.config import DATA_ROOT, RAW_DIR
print("DATA_ROOT:", DATA_ROOT)
print("RAW_DIR:", RAW_DIR)

# 5. Probar carga
quarters = get_quarters()
print(f"Trimestres: {len(quarters)}")

df = load_quarter(quarters[0], "FTS")
print(df.shape)



DATA_ROOT: D:\financial_risk_data
RAW_DIR: D:\financial_risk_data\dataraw
Trimestres: 40
(6193, 3369)


Veamos que estructura tienen los archivos de un trimestre completo, i.e., cuántas filas tiene cada archivo, cuántas columnas, si falta algun archivo.

In [ ]:
import numpy as np
import pandas as pd


quarters = get_quarters()

file_types = ["FTS", "CDI", "RAT", "MERG", "STRU"]

shapes = []

for q in quarters:
    suffix = q.name[3:]
    for ft in file_types:
        try:
            df = load_quarter(q, ft)
            shapes.append({
                "quarter": suffix,
                "file": ft,
                "rows": df.shape[0],
                "cols": df.shape[1]
            })
        except FileNotFoundError:
            shapes.append({
                "quarter": suffix,
                "file": ft,
                "rows": None,
                "cols": None
            })

pd.DataFrame(shapes)

,quarter,file,rows,cols
0,1603,FTS,6193,3369
1,1603,CDI,6193,1097
2,1603,RAT,6193,253
3,1603,MERG,26336,61
4,1603,STRU,6546,122
...,...,...,...,...
195,2512,FTS,4408,3369
196,2512,CDI,4408,1097
197,2512,RAT,4408,253
198,2512,MERG,28422,61


El resultado es que tenemos 200 filas (40 trimestres por 5 archivos de datos)

In [ ]:
# el número total de filas sumando las filas de todos los trimestres
def total_rows(file_type):
    total = 0
    for q in get_quarters():
        try:
            df = load_quarter(q, file_type)
            total += len(df)
        except FileNotFoundError:
            pass
    return total

totals = {ft: total_rows(ft) for ft in file_types}
totals

{'FTS': 206129, 'CDI': 206129, 'RAT': 206129, 'MERG': 1100520, 'STRU': 219576}

In [6]:
def validate_files():
    quarters = get_quarters()
    missing = []

    for q in quarters:
        suffix = q.name[3:]
        for ft in file_types:
            expected = q / f"{ft}{suffix}.csv"
            if not expected.exists():
                missing.append(str(expected))

    return missing

missing_files = validate_files()
missing_files

[]

In [7]:
def explore_columns():
    sample_quarter = get_quarters()[0]
    info = {}

    for ft in file_types:
        df = load_quarter(sample_quarter, ft)
        info[ft] = list(df.columns)

    return info

columns_info = explore_columns()
columns_info


{'FTS': ['AASLRIND',
  'AATOTFV',
  'ABCUBK',
  'ABCUOTH',
  'ABCXBK',
  'ABCXOTH',
  'ACEPT',
  'ACEPTDOM',
  'ACEPTFOR',
  'ACLPCDA',
  'AGLOSEND',
  'ALLCRCD',
  'ALNSCHTM',
  'ANNUASST',
  'ANNUSALE',
  'APCDLNLS',
  'APCDOFA',
  'APCDSCHM',
  'ASCEAUTO',
  'ASCECI',
  'ASCECONS',
  'ASCECRCD',
  'ASCEHEL',
  'ASCEOTH',
  'ASCERES',
  'ASDRAUTO',
  'ASDRCI',
  'ASDRCONS',
  'ASDRCRCD',
  'ASDRHEL',
  'ASDROTH',
  'ASDRRES',
  'ASDRTL',
  'ASSET',
  'ASSET0',
  'ASSET10',
  'ASSET100',
  'ASSET150',
  'ASSET20',
  'ASSET250',
  'ASSET300',
  'ASSET4',
  'ASSET400',
  'ASSET50',
  'ASSET600',
  'ASSET625',
  'ASSET937',
  'ASSETAG',
  'ASSETCON',
  'ASSETDOM',
  'ASSETELG',
  'ASSETFOR',
  'ASSETIBF',
  'ASSETMNT',
  'ASSETPLG',
  'ASSETRE',
  'ASSETREC',
  'ASSETTFR',
  'ASSETTR',
  'ASSLD100',
  'ASSLDCEA',
  'ASSLDRCR',
  'ASSNETDD',
  'ASST1250',
  'ASST2',
  'ASTCRSUB',
  'ASTNETTM',
  'ASTSERV',
  'ATOTFV',
  'ATOTL1FV',
  'ATOTL2FV',
  'ATOTL3FV',
  'ATOTNTFV',
  'ATRR',
  'AT